# data rectify

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd

DATASET_DIR = Path("PATH/TO/DATASET")
BINNED_DIR = DATASET_DIR / "binned_outputs" / "square_008um"
IMAGE_FILE = DATASET_DIR / "tissue_image.tif"
SCALEFACTORS_FILE = BINNED_DIR / "spatial" / "scalefactors_json.json"
GENE_NAMES_FILE = DATASET_DIR / "gene-names.txt"
SCRIPT_DIR = Path.cwd()  # Open this notebook from its VisiumHD folder.
RECTIFIED_DIR = DATASET_DIR / "data_rectified"

with SCALEFACTORS_FILE.open(encoding="utf-8") as handle:
    json_data = json.load(handle)
for key in ("spot_diameter_fullres", "microns_per_pixel"):
    if key not in json_data:
        raise KeyError(f"Missing {key} in {SCALEFACTORS_FILE}; provide the original dataset metadata")

subprocess.run([
    sys.executable, str(SCRIPT_DIR / "ViHD_rectify.py"),
    "--spot_file", str(BINNED_DIR / "spatial" / "tissue_positions.parquet"),
    "--image_file", str(IMAGE_FILE),
    "--output_dir", str(RECTIFIED_DIR),
], check=True)


# Resolution simulation

In [ ]:
if not GENE_NAMES_FILE.is_file():
    raise FileNotFoundError("Supply the finalized study SVG list before resolution simulation")

spot_sizes = {
    "dense": list(range(16, 150, 8)),
    "proportional": list(range(16, 150, 8)),
    "fixed_spotSize": list(range(56, 210, 8)),
    "fixed_spotDistance": list(range(16, 100, 8)),
}
MODES = tuple(spot_sizes)  # Restrict this explicitly for an individual simulation.
MATRIX_ROOT = DATASET_DIR / "raw_resolution" / "matrix"
tissue_positions = RECTIFIED_DIR / "tissue_positions_adj.parquet"

for mode in MODES:
    parameter_name = "spotDistance" if mode == "fixed_spotSize" else "spotSize"
    for value in spot_sizes[mode]:
        output_dir = MATRIX_ROOT / mode / f"{parameter_name}_{value}"
        subprocess.run([
            sys.executable, str(SCRIPT_DIR / "ViHD_preprocessing.py"),
            "--mode", mode, "--spot_size", str(value),
            "--tissue_positions", str(tissue_positions),
            "--matrix_dir", str(BINNED_DIR),
            "--output_dir", str(output_dir),
            "--gene_names", str(GENE_NAMES_FILE),
        ], check=True)


In [ ]:
for mode in MODES:
    parameter_name = "spotDistance" if mode == "fixed_spotSize" else "spotSize"
    for value in spot_sizes[mode]:
        diameter_um = 56 if mode == "fixed_spotSize" else value
        data_dir = MATRIX_ROOT / mode / f"{parameter_name}_{value}"
        output_prefix = f"{mode}_{diameter_um}um"
        diameter_px = float(json_data["spot_diameter_fullres"]) * diameter_um / 8
        (data_dir / "radius-raw.txt").write_text(str(diameter_px / 2))
        (data_dir / "radius.txt").write_text(str(round(diameter_px / 2)))
        (data_dir / "pixel-size-raw.txt").write_text(str(json_data["microns_per_pixel"]))
        cnts = pd.read_csv(data_dir / f"cnts_{output_prefix}.tsv", sep="\t")
        locs = pd.read_csv(data_dir / f"locs_{output_prefix}.tsv", sep="\t")
        cnts.to_csv(data_dir / "cnts.tsv", sep="\t", index=False)
        locs.to_csv(data_dir / "locs-raw.tsv", sep="\t", index=False)
        cnts.to_csv(data_dir / "cnts.csv", index=False)
        locs.to_csv(data_dir / "locs-raw.csv", index=False)
